In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

# Policy network
class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, state):
        return self.fc(state)

# Environment and setup
env = gym.make("CartPole-v1")
policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
optimizer = optim.Adam(policy.parameters(), lr=1e-2)
gamma = 0.99

for episode in range(500):
    states, actions, rewards = [], [], []
    state = env.reset()[0]
    done = False

    # 1️⃣ Run an episode
    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32)
        probs = policy(state_tensor)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()

        next_state, reward, done, _, _ = env.step(action.item())
        states.append(state_tensor)
        actions.append(action)
        rewards.append(reward)
        state = next_state

    # 2️⃣ Compute returns (discounted rewards)
    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # normalize

    # 3️⃣ Compute policy loss
    log_probs = []
    for state, action in zip(states, actions):
        probs = policy(state)
        dist = torch.distributions.Categorical(probs)
        log_probs.append(dist.log_prob(action))
    log_probs = torch.stack(log_probs)

    loss = -(log_probs * returns).sum()

    # 4️⃣ Backpropagate
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Episode {episode}, total reward: {sum(rewards):.2f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

# --- Policy Network ---
class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, state):
        return self.net(state)

# --- REINFORCE Agent ---
def reinforce(env_name="CartPole-v1", lr=1e-3, gamma=0.99, episodes=500):
    env = gym.make(env_name)
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    for ep in range(episodes):
        log_probs, rewards = [], []
        state = env.reset()[0] if isinstance(env.reset(), tuple) else env.reset()

        done = False
        while not done:
            state_t = torch.tensor(state, dtype=torch.float32)
            probs = policy(state_t)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))

            next_state, reward, done, *_ = env.step(action.item())
            rewards.append(reward)
            state = next_state

        # Compute discounted returns
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # Compute policy gradient loss
        loss = -torch.sum(torch.stack(log_probs) * returns)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Episode {ep+1}: total reward = {sum(rewards)}")

    env.close()

reinforce()
